# WF2020 Short-Volshock Target Enrichment

Generate fold-local short-volshock de-risk labels for the `wf_2020_train` SOXLBacktest target contract using only the fold training window feature distribution. This is a target-generation card only; it does not train, promote, or retain an actor.

In [1]:

from __future__ import annotations

import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
os.chdir(ROOT)

from src.soxl_quant.app.state_objective_training_targets import (
    ShortVolshockDeRiskTargetConfig,
    apply_short_volshock_de_risk_target_gate,
    summarize_short_volshock_de_risk_targets,
)

EXPERIMENT = 'wf2020_short_volshock_target_enrichment_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_TARGET_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_wf2020_soxlbacktest_target_contract_v1' / 'soxlbacktest_wf2020_target_contract.csv'
FEATURE_FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')

source = pd.read_csv(SOURCE_TARGET_PATH)
source['datetime'] = pd.to_datetime(source['datetime'])
features = pd.read_pickle(FEATURE_FRAME_PATH).reset_index().rename(columns={'index': 'datetime'})
features['datetime'] = pd.to_datetime(features['datetime'])
train_start = pd.Timestamp(source['train_start'].dropna().iloc[0])
train_end = pd.Timestamp(source['train_end'].dropna().iloc[0])
eval_start = pd.Timestamp(source['eval_start'].dropna().iloc[0])
eval_end = pd.Timestamp(source['eval_end'].dropna().iloc[0])
train_features = features[(features['datetime'] >= train_start) & (features['datetime'] <= train_end)].copy()
config = ShortVolshockDeRiskTargetConfig(target_cap=0.05, min_feature_hits=2, min_position_delta=0.02)
thresholds = pd.DataFrame([
    {
        'fold': 'wf_2020',
        'realized_vol_21_q80': train_features['realized_vol_21'].quantile(0.80),
        'vol_of_vol_21_q80': train_features['vol_of_vol_21'].quantile(0.80),
        'downside_semivol_21_q80': train_features['downside_semivol_21'].quantile(0.80),
        'threshold_source_start': train_features['datetime'].min().date().isoformat(),
        'threshold_source_end': train_features['datetime'].max().date().isoformat(),
        'threshold_source_rows': int(len(train_features)),
    }
])

enriched = apply_short_volshock_de_risk_target_gate(source, features, thresholds, config=config)
enriched['target_scope'] = 'fold_local'
enriched['source_label'] = 'wf2020_soxlbacktest_short_volshock_enrichment'
enriched['source_target_path'] = str(SOURCE_TARGET_PATH.relative_to(ROOT))
summary_core = summarize_short_volshock_de_risk_targets(enriched)

eval_overlap = enriched[(enriched['datetime'] >= eval_start) & (enriched['datetime'] <= eval_end)]
outside_train = enriched[(enriched['datetime'] < train_start) | (enriched['datetime'] > train_end)]
feature_missing_rows = int(enriched[['realized_vol_21', 'vol_of_vol_21', 'downside_semivol_21']].isna().any(axis=1).sum())
scope_audit = pd.DataFrame([
    {
        'fold': 'wf_2020_train',
        'train_start': train_start.date().isoformat(),
        'train_end': train_end.date().isoformat(),
        'eval_start': eval_start.date().isoformat(),
        'eval_end': eval_end.date().isoformat(),
        'target_rows': int(len(enriched)),
        'eval_overlap_rows': int(len(eval_overlap)),
        'outside_train_rows': int(len(outside_train)),
        'feature_missing_rows': feature_missing_rows,
        'short_volshock_state_rows': int(enriched['short_volshock_state'].fillna(False).astype(bool).sum()),
        'short_volshock_target_rows': int(enriched['short_volshock_target_state'].fillna(False).astype(bool).sum()),
        'target_scope_gate': bool(len(eval_overlap) == 0 and len(outside_train) == 0),
    }
])

thresholds.to_csv(OUT_DIR / 'wf2020_short_volshock_thresholds.csv', index=False)
enriched.to_csv(OUT_DIR / 'wf2020_short_volshock_target_contract.csv', index=False)
scope_audit.to_csv(OUT_DIR / 'target_scope_audit.csv', index=False)

summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'wf2020_short_volshock_target_enrichment_ready_for_bc_screen' if summary_core['target_state_rows'] > 0 and bool(scope_audit.iloc[0]['target_scope_gate']) else 'wf2020_short_volshock_target_enrichment_blocked',
    'source_target_path': str(SOURCE_TARGET_PATH.relative_to(ROOT)),
    'feature_frame': str(FEATURE_FRAME_PATH.relative_to(ROOT)),
    'target_path': str((OUT_DIR / 'wf2020_short_volshock_target_contract.csv').relative_to(ROOT)),
    'threshold_path': str((OUT_DIR / 'wf2020_short_volshock_thresholds.csv').relative_to(ROOT)),
    'scope_audit_path': str((OUT_DIR / 'target_scope_audit.csv').relative_to(ROOT)),
    'config': config.__dict__,
    'thresholds': thresholds.to_dict('records')[0],
    'target_summary': summary_core,
    'scope_audit': scope_audit.to_dict('records')[0],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'next_mechanism': 'rerun_short_volshock_three_fold_bc_slice_screen_with_wf2020_enriched_targets',
    'sidecars': ['wf2020_short_volshock_thresholds.csv', 'wf2020_short_volshock_target_contract.csv', 'target_scope_audit.csv', 'selection.json', 'summary.json', 'conclusion.md'],
}
selection = {k: summary[k] for k in ['experiment', 'created_at', 'decision', 'target_path', 'threshold_path', 'promotion_allowed', 'actor_retention_allowed', 'next_mechanism']}
write_json(OUT_DIR / 'selection.json', selection)
write_json(OUT_DIR / 'summary.json', summary)

conclusion = f"""
## Conclusion

Decision: `{summary['decision']}`.

Generated `wf_2020_train` short-volshock enrichment from the fold-local training feature window `{thresholds.iloc[0]['threshold_source_start']}` to `{thresholds.iloc[0]['threshold_source_end']}`. Thresholds are q80 values for realized volatility, vol-of-vol, and downside semivolatility.

Target rows: `{len(enriched)}`; short-volshock state rows: `{summary_core['short_volshock_state_rows']}`; target cap rows: `{summary_core['target_state_rows']}`; max cap delta: `{summary_core['max_risk_cap_delta']:.4f}`.

Scope audit: `eval_overlap_rows={int(scope_audit.iloc[0]['eval_overlap_rows'])}`, `outside_train_rows={int(scope_audit.iloc[0]['outside_train_rows'])}`, `target_scope_gate={bool(scope_audit.iloc[0]['target_scope_gate'])}`. Feature-missing rows are `{feature_missing_rows}` because the current feature frame starts after the earliest legacy SOXLBacktest target rows; missing features do not create short-volshock state rows.

This card is target-generation only: `promotion_allowed=false`, `actor_retention_allowed=false`. Next action: `{summary['next_mechanism']}`.
"""
(OUT_DIR / 'conclusion.md').write_text(conclusion.strip() + '\n')
display(Markdown(conclusion))



## Conclusion

Decision: `wf2020_short_volshock_target_enrichment_ready_for_bc_screen`.

Generated `wf_2020_train` short-volshock enrichment from the fold-local training feature window `2011-03-11` to `2019-12-31`. Thresholds are q80 values for realized volatility, vol-of-vol, and downside semivolatility.

Target rows: `2470`; short-volshock state rows: `405`; target cap rows: `390`; max cap delta: `0.6918`.

Scope audit: `eval_overlap_rows=0`, `outside_train_rows=0`, `target_scope_gate=True`. Feature-missing rows are `253` because the current feature frame starts after the earliest legacy SOXLBacktest target rows; missing features do not create short-volshock state rows.

This card is target-generation only: `promotion_allowed=false`, `actor_retention_allowed=false`. Next action: `rerun_short_volshock_three_fold_bc_slice_screen_with_wf2020_enriched_targets`.
